In [ ]:
import re

from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "retail_operations_dev")
catalog = dbutils.widgets.get("catalog").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):
    raise ValueError("Invalid catalog name.")

inventory_source_path = (
    f"/Volumes/{catalog}/bronze/supplier_files/inventory"
)

dbutils.fs.mkdirs(inventory_source_path)

print("Upload inventory CSV files into:")
print(inventory_source_path)

In [ ]:
inventory_files = dbutils.fs.ls(inventory_source_path)

for file_info in inventory_files:
    print(
        f"{file_info.name}: {file_info.size} bytes"
    )

In [ ]:
from pyspark.sql import types as T

inventory_raw_schema = T.StructType([
    T.StructField("warehouse_id", T.StringType(), True),
    T.StructField("product_id", T.StringType(), True),
    T.StructField("on_hand_quantity", T.StringType(), True),
    T.StructField("reserved_quantity", T.StringType(), True),
    T.StructField("snapshot_at", T.StringType(), True),
    T.StructField("_rescued_data", T.StringType(), True),
])


inventory_state_path = (
    f"/Volumes/{catalog}/operations/streaming_checkpoints/"
    "inventory_v1"
)

inventory_schema_path = f"{inventory_state_path}/schema"
inventory_checkpoint_path = f"{inventory_state_path}/checkpoint"

inventory_bronze_table = f"{catalog}.bronze.inventory_raw"

In [ ]:
inventory_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format" , "csv")
    .option("cloudFiles.schemaLocation" , inventory_schema_path)
    .option("header" , "true")
    .option("pathGlobFilter" , "*.csv")
    .option("cloudFiles.schemaEvolutionMode" , "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(inventory_raw_schema)
    .load(inventory_source_path)
    .select(
        "*",
        F.col("_metadata.file_path").alias("source_file")
    )
    .withColumn(
        "ingested_at",
        F.current_timestamp()
    )
)

print("Streaming DataFrame:", inventory_stream.isStreaming)

In [ ]:
inventory_query = (
    inventory_stream
    .writeStream
    .format("delta")
    .option("checkpointLocation" , inventory_checkpoint_path)
    .outputMode("append")
    .trigger(availableNow = True)
    .toTable(inventory_bronze_table)
)

inventory_query.awaitTermination()

print(f"Inventory ingestion completed: {inventory_bronze_table}")

In [ ]:
saved_inventory = spark.table(inventory_bronze_table)

display(
    saved_inventory.orderBy(
        "warehouse_id",
        "product_id",
        "snapshot_at"
    )
)

display(
    saved_inventory
    .groupBy("source_file")
    .count()
)

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

parsed_inventory = (
    spark.table(inventory_bronze_table)
    .withColumnRenamed(
        "on_hand_quantity",
        "on_hand_quantity_raw"
    )
    .withColumnRenamed(
        "reserved_quantity",
        "reserved_quantity_raw"
    )
    .withColumnRenamed(
        "snapshot_at",
        "snapshot_at_raw"
    )
    .withColumn(
        "on_hand_quantity",
        F.expr("try_cast(on_hand_quantity_raw AS BIGINT)")
    )
    .withColumn(
        "reserved_quantity",
        F.expr("try_cast(reserved_quantity_raw AS BIGINT)")
    )
    .withColumn(
        "snapshot_at",
        F.try_to_timestamp("snapshot_at_raw")
    )
)

display(parsed_inventory)

In [ ]:
def has_text(column_name):
    return F.coalesce(
        F.length(F.trim(F.col(column_name))) > 0,
        F.lit(False)
    )


inventory_rules = [
    (
        has_text("warehouse_id"),
        "Missing warehouse_id"
    ),
    (
        has_text("product_id"),
        "Missing product_id"
    ),
    (
        F.col("on_hand_quantity_raw").rlike(r"^[0-9]+$")
        & F.col("on_hand_quantity").isNotNull(),
        "on_hand_quantity must be a non-negative whole number within BIGINT range"
    ),
    (
        F.col("reserved_quantity_raw").rlike(r"^[0-9]+$")
        & F.col("reserved_quantity").isNotNull(),
        "reserved_quantity must be a non-negative whole number within BIGINT range"
    ),
    (
        F.col("reserved_quantity") <= F.col("on_hand_quantity"),
        "reserved_quantity exceeds on_hand_quantity"
    ),
    (
        F.col("snapshot_at").isNotNull(),
        "Missing or invalid snapshot_at"
    ),
    (
        F.col("_rescued_data").isNull(),
        "Unexpected inventory data was rescued during ingestion"
    ),
]

checked_inventory = (
    parsed_inventory
    .withColumn(
        "rejection_reasons",
        F.filter(
            F.array(*[
                F.when(
                    ~F.coalesce(condition, F.lit(False)),
                    F.lit(reason)
                )
                for condition, reason in inventory_rules
            ]),
            lambda reason: reason.isNotNull()
        )
    )
    .withColumn(
        "is_valid",
        F.size("rejection_reasons") == 0
    )
)

display(
    checked_inventory.select(
        "warehouse_id",
        "product_id",
        "on_hand_quantity",
        "reserved_quantity",
        "snapshot_at",
        "is_valid",
        "rejection_reasons"
    ).orderBy("warehouse_id", "product_id")
)

In [ ]:
valid_inventory = checked_inventory.filter(
    F.col("is_valid")
)

rejected_inventory = checked_inventory.filter(
    ~F.col("is_valid")
)

display(
    checked_inventory.groupBy("is_valid").count()
)

display(
    rejected_inventory.select(
        "warehouse_id",
        "product_id",
        "on_hand_quantity_raw",
        "reserved_quantity_raw",
        "snapshot_at_raw",
        "rejection_reasons",
        "source_file"
    )
)

assert rejected_inventory.limit(1).count() == 0, (
    "Inventory validation failed. Review rejected_inventory "
    "before publishing current stock."
)

In [ ]:
inventory_version_checks = (
    valid_inventory
    .groupBy(
        "warehouse_id",
        "product_id",
        "snapshot_at"
    )
    .agg(
        F.countDistinct(
            F.struct(
                "on_hand_quantity",
                "reserved_quantity"
            )
        ).alias("quantity_versions")
    )
)

inventory_conflicts = inventory_version_checks.filter(
    F.col("quantity_versions") > 1
)

if inventory_conflicts.limit(1).count() > 0:
    display(inventory_conflicts)

    raise ValueError(
        "Conflicting inventory quantities share the same "
        "warehouse_id, product_id, and snapshot_at. "
        "Resolve them before publishing current stock."
    )

print("Inventory snapshot conflict checks passed.")

display(inventory_version_checks)

In [ ]:
from pyspark.sql.window import Window

inventory_window = (
    Window
    .partitionBy("warehouse_id", "product_id")
    .orderBy(
        F.col("snapshot_at").desc(),
        F.col("ingested_at").asc(),
        F.col("source_file").asc()
    )
)

current_inventory = (
    valid_inventory
    .withColumn(
        "snapshot_rank",
        F.row_number().over(inventory_window)
    )
    .filter(F.col("snapshot_rank") == 1)
    .select(
        "warehouse_id",
        "product_id",
        "on_hand_quantity",
        "reserved_quantity",
        "snapshot_at",
        "source_file",
        "ingested_at"
    )
    .withColumn(
        "available_quantity",
        F.col("on_hand_quantity") - F.col("reserved_quantity")
    )
)

display(
    current_inventory.orderBy(
        "warehouse_id",
        "product_id"
    )
)

In [ ]:
from delta.tables import DeltaTable

inventory_silver_table = (
    f"{catalog}.silver.inventory_current"
)

current_inventory.createOrReplaceTempView(
    "current_inventory_source"
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {inventory_silver_table}
    USING DELTA
    AS
    SELECT *
    FROM current_inventory_source
    WHERE 1 = 0
""")

(
    DeltaTable.forName(spark, inventory_silver_table)
    .alias("target")
    .merge(
        current_inventory.alias("source"),
        """
        target.warehouse_id = source.warehouse_id
        AND target.product_id = source.product_id
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete()
    .execute()
)

print(f"Saved: {inventory_silver_table}")

In [ ]:
saved_inventory_current = (
    spark.table(inventory_silver_table)
    .select(*current_inventory.columns)
)

assert (
    current_inventory.exceptAll(saved_inventory_current)
    .limit(1).count() == 0
), "Some prepared inventory rows are missing or different in Silver."

assert (
    saved_inventory_current.exceptAll(current_inventory)
    .limit(1).count() == 0
), "Silver contains extra or different inventory rows."

display(
    saved_inventory_current.agg(
        F.count("*").alias("stock_positions"),
        F.countDistinct(
            "warehouse_id",
            "product_id"
        ).alias("unique_stock_positions")
    )
)

print("Current-inventory checks passed.")

In [ ]:
inventory = spark.table(
    f"{catalog}.silver.inventory_current"
)

products = spark.table(
    f"{catalog}.silver.products_current"
)

duplicate_products = (
    products
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)

assert duplicate_products.limit(1).count() == 0, (
    "Duplicate product IDs would multiply inventory rows."
)

In [ ]:
inventory_enriched = (
    inventory.alias("stock")
    .join(
        products.alias("products"),
        F.col("stock.product_id") == F.col("products.product_id"),
        "left"
    )
    .select(
        "stock.*",
        F.col("products.product_name").alias("current_product_name"),
        F.col("products.category").alias("current_category"),
        F.col("products.is_active").alias("product_currently_active"),
        F.col("products.updated_at").alias("product_catalog_updated_at"),
        F.col("products.product_id").isNull().alias("product_missing")
    )
    .withColumn(
        "stock_status",
        F.when(
            F.col("available_quantity") == 0,
            F.lit("unavailable")
        ).otherwise(
            F.lit("available")
        )
    )
)

display(
    inventory_enriched.orderBy(
        "warehouse_id",
        "product_id"
    )
)

In [ ]:
inventory_fields_after_join = inventory_enriched.select(
    *inventory.columns
)

assert (
    inventory.exceptAll(inventory_fields_after_join)
    .limit(1).count() == 0
), "Some inventory positions were lost or changed."

assert (
    inventory_fields_after_join.exceptAll(inventory)
    .limit(1).count() == 0
), "Product enrichment introduced extra or changed inventory positions."

display(
    inventory_enriched
    .groupBy("product_missing")
    .count()
)

In [ ]:
inventory_gold_table = (
    f"{catalog}.gold.inventory_availability"
)

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold"
)

(
    inventory_enriched
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(inventory_gold_table)
)

saved_inventory_gold = (
    spark.table(inventory_gold_table)
    .select(*inventory_enriched.columns)
)

assert (
    inventory_enriched.exceptAll(saved_inventory_gold)
    .limit(1).count() == 0
), "Some prepared inventory rows are missing or different in Gold."

assert (
    saved_inventory_gold.exceptAll(inventory_enriched)
    .limit(1).count() == 0
), "Gold contains extra or different inventory rows."

print(f"Published and verified: {inventory_gold_table}")

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.inventory_freshness AS
    SELECT
        warehouse_id,
        product_id,
        current_product_name,
        available_quantity,
        stock_status,
        snapshot_at,
        ingested_at,
        CASE
            WHEN snapshot_at IS NULL
                THEN 'unknown'
            WHEN snapshot_at > current_timestamp()
                THEN 'future_timestamp'
            WHEN snapshot_at < current_timestamp() - INTERVAL 48 HOURS
                THEN 'stale'
            ELSE 'fresh'
        END AS freshness_status
    FROM {catalog}.gold.inventory_availability
""")

display(
    spark.table(f"{catalog}.gold.inventory_freshness")
    .orderBy("freshness_status", "warehouse_id", "product_id")
)

In [ ]:
inventory_freshness = spark.table(
    f"{catalog}.gold.inventory_freshness"
)

display(
    inventory_freshness
    .groupBy("freshness_status")
    .count()
)

display(
    inventory_freshness
    .filter(F.col("freshness_status") != "fresh")
    .orderBy("snapshot_at", "warehouse_id", "product_id")
)